# 02 Validation: choosing a gap-filling method

Real readings are hidden, filled, and compared with the true value. The data is split **by time** so the choice is not tuned on the test period: Feb–May = development (method selection), June = test (used once).

In [1]:
import os
import pandas as pd
from mma3001_project.reconstruction import evaluate_methods, summarise_errors, bootstrap_rmse

clean = pd.read_csv("../data/processed/sensor869_clean.csv", index_col=0, parse_dates=True)
co2 = clean["co2_ppm"]
test_start = pd.Timestamp("2024-06-01", tz="Australia/Melbourne")   # Feb–May dev | June test
dev = co2[co2.index < test_start]
test = co2[co2.index >= test_start]
print(len(dev), "development slots,", len(test), "test slots")

13296 development slots, 1895 test slots


## 1. Compare methods on development data
200 artificial gaps per length; lengths 1 and 2 slots match the real gaps found in 01. LOCF is the baseline.

In [2]:
dev_table = summarise_errors(evaluate_methods(dev, lengths=[1, 2], n_gaps=200, seed=1))   # 1-2 slots = real gap lengths (01)
dev_table.round(1)

RMSE   MAE  bias    n
method length                       
cubic  1       12.5   9.8  -0.8  200
       2       14.1  11.1  -1.2  400
linear 1       10.4   8.2  -0.6  200
       2       10.8   8.4  -0.4  400
locf   1       12.3   9.6  -0.7  200
       2       13.3  10.0   0.4  400
pchip  1       10.6   8.4  -0.6  200
       2       11.2   8.7  -0.5  400

**Result:** linear has the lowest RMSE at both lengths (10.4, 10.8 ppm). The cubic spline overshoots and is worse than LOCF for 2-slot gaps. **Linear is selected.**

## 2. One-time test on June
Linear vs the LOCF baseline. 95 % bootstrap intervals resample whole gaps, since errors within a gap are correlated.

In [3]:
test_errors = evaluate_methods(test, lengths=[1, 2], n_gaps=200, seed=2, methods=("locf", "linear"))
for m in ["locf", "linear"]:
    r, lo, hi = bootstrap_rmse(test_errors[test_errors["method"] == m])
    print(f"{m:6s}  test RMSE = {r:4.1f} ppm  (95% CI {lo:.1f} to {hi:.1f})")

locf    test RMSE = 11.0 ppm  (95% CI 10.3 to 11.7)
linear  test RMSE =  9.7 ppm  (95% CI 9.1 to 10.3)


**Result:** linear 9.7 ppm vs LOCF 11.0 ppm on unseen data; the intervals barely touch (10.3), so the improvement holds outside the development period.

**Limits:** only gaps ≤ 20 min are validated; longer outages would need re-testing. Errors are relative to the sensor reading, not true room CO₂.

## 3. Save

In [4]:
os.makedirs("../results", exist_ok=True)
dev_table.to_csv("../results/reconstruction_development.csv")
summarise_errors(test_errors).to_csv("../results/reconstruction_test.csv")